# Export Sessions to Excel

This notebook exports remote viewing session data from the Social RV Research API into Excel spreadsheets.

**Features:**
- Fetches all sessions from the API (handles pagination automatically)
- Flattens nested data into individual columns
- Optionally embeds target images directly in the spreadsheet
- Creates separate sheets for sessions and targets

**Requirements:**
- A valid Research API key in your `.env` file


In [ ]:
# Setup and Dependencies
import sys
sys.path.insert(0, '../src')

import os
import requests
from pathlib import Path
from datetime import datetime
from io import BytesIO
from typing import Optional

from dotenv import load_dotenv
load_dotenv('../.env')

# Excel manipulation
from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, PatternFill

# Image processing
from PIL import Image as PILImage

# Our API client
from comparative_judging import SocialRVClient

# Output directory
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR.absolute()}")


In [ ]:
# Configuration

# Maximum number of sessions to fetch (set to None for all sessions)
MAX_SESSIONS = 100  # e.g., 100 for testing, None for all

# Include options
INCLUDE_UNSUBMITTED = False  # Include sessions not yet submitted
INCLUDE_NON_PUBLIC = True    # Include private sessions
INCLUDE_LOW_VALUE = False    # Include low-value sessions

# Image settings
EMBED_IMAGES = True         # Set to False for faster export without images
MAX_IMAGE_WIDTH = 200       # Maximum width in pixels
MAX_IMAGE_HEIGHT = 200      # Maximum height in pixels

print(f"📋 Configuration:")
print(f"   Max sessions: {MAX_SESSIONS if MAX_SESSIONS else 'All'}")
print(f"   Include unsubmitted: {INCLUDE_UNSUBMITTED}")
print(f"   Include non-public: {INCLUDE_NON_PUBLIC}")
print(f"   Include low-value: {INCLUDE_LOW_VALUE}")
print(f"   Embed images: {EMBED_IMAGES}")
print(f"   Max image size: {MAX_IMAGE_WIDTH}x{MAX_IMAGE_HEIGHT}px")


In [ ]:
# Initialize the API client and fetch sessions
client = SocialRVClient()

def progress(current, total):
    print(f"\r⏳ Fetching: {current}/{total} sessions", end="")

print("📥 Fetching sessions from API...")
sessions = client.fetch_all_sessions(
    include_unsubmitted=INCLUDE_UNSUBMITTED,
    include_non_public=INCLUDE_NON_PUBLIC,
    include_low_value=INCLUDE_LOW_VALUE,
    max_sessions=MAX_SESSIONS,
    progress_callback=progress
)

print(f"\n✅ Fetched {len(sessions)} sessions")


In [ ]:
# Helper functions for image handling

def download_image(url: str) -> Optional[bytes]:
    """Download an image from a URL. Returns bytes or None if download fails."""
    if not url:
        return None
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return response.content
    except Exception as e:
        print(f"⚠️  Failed to download image: {e}")
        return None


def create_scaled_xl_image(image_bytes: bytes, max_width: int, max_height: int) -> Optional[XLImage]:
    """Create an openpyxl Image object scaled to fit within bounds."""
    try:
        # Load image with PIL to handle format conversion
        pil_img = PILImage.open(BytesIO(image_bytes))
        orig_width, orig_height = pil_img.size
        
        # Convert to PNG if not already a supported format
        img_format = pil_img.format
        if img_format not in ('PNG', 'JPEG', 'GIF'):
            png_buffer = BytesIO()
            if pil_img.mode in ('RGBA', 'LA', 'P'):
                pil_img = pil_img.convert('RGBA')
            else:
                pil_img = pil_img.convert('RGB')
            pil_img.save(png_buffer, format='PNG')
            png_buffer.seek(0)
            image_bytes = png_buffer.getvalue()
        
        # Calculate scale factor
        width_scale = max_width / orig_width
        height_scale = max_height / orig_height
        scale = min(width_scale, height_scale, 1.0)  # Don't upscale
        
        # Create XL image
        xl_img = XLImage(BytesIO(image_bytes))
        xl_img.width = int(orig_width * scale)
        xl_img.height = int(orig_height * scale)
        
        return xl_img
    except Exception as e:
        print(f"⚠️  Failed to create image: {e}")
        return None


print("✅ Helper functions defined")


In [ ]:
# Flatten session data for spreadsheet

def flatten_session(session) -> dict:
    """Flatten a session's nested data into a flat dictionary."""
    decoy_ids_str = ', '.join(session.decoy_ids) if session.decoy_ids else ''
    
    return {
        'session_id': session.id,
        'user_id': session.user_id,
        'user_display_name': session.user_display_name,
        'tasking_time': session.tasking_time,
        'submission_time': session.submission_time,
        'target_coordinate': session.target_coordinate,
        'weekly_target_id': session.weekly_target_id,
        'is_public': session.is_public,
        'is_low_value': session.is_low_value,
        'is_blockchain_verified': session.is_blockchain_verified,
        'self_score': session.self_score,
        'community_score_avg': session.community_score_average,
        'community_score_count': session.community_score_count,
        'cj_rank': session.comparative_judging_rank,
        'p_value': session.p_value,
        'rank': session.rank,
        'rank_denominator': session.rank_denominator,
        'z_score': session.z_score,
        'vector_text_similarity': session.vector_text_similarity,
        'num_comments': session.num_comments,
        'decoy_ids': decoy_ids_str,
        'target_id': session.target_id,
        'target_description': session.target_description,
        'target_image_url': session.target_image_url,
        'session_media_count': len(session.session_media_urls),
    }


# Flatten all sessions
flattened_sessions = [flatten_session(s) for s in sessions]

print(f"✅ Flattened {len(flattened_sessions)} sessions")

# Preview flattened data
if flattened_sessions:
    print("\n📋 Sample flattened data:")
    sample = flattened_sessions[0]
    for key, value in list(sample.items())[:10]:
        print(f"   {key}: {str(value)[:60]}{'...' if len(str(value or '')) > 60 else ''}")


In [ ]:
# Create Excel workbook

print("📝 Creating Excel workbook...")

wb = Workbook()
ws = wb.active
ws.title = "Sessions"

# Styles
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="4F46E5", end_color="4F46E5", fill_type="solid")  # Indigo
header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Define columns
DATA_COLUMNS = [
    ('session_id', 'Session ID', 20),
    ('user_id', 'User ID', 20),
    ('user_display_name', 'Display Name', 20),
    ('tasking_time', 'Tasking Time', 20),
    ('submission_time', 'Submission Time', 20),
    ('target_coordinate', 'Target Coordinate', 18),
    ('weekly_target_id', 'Weekly Target ID', 18),
    ('is_public', 'Is Public', 10),
    ('is_low_value', 'Is Low Value', 12),
    ('is_blockchain_verified', 'Blockchain Verified', 18),
    ('self_score', 'Self Score', 12),
    ('community_score_avg', 'Community Score Avg', 18),
    ('community_score_count', 'Community Score Count', 20),
    ('cj_rank', 'CJ Rank', 10),
    ('p_value', 'P-Value', 12),
    ('rank', 'AI Rank', 10),
    ('z_score', 'Z-Score', 12),
    ('num_comments', 'Comments', 10),
    ('decoy_ids', 'Decoy IDs', 40),
    ('target_id', 'Target ID', 20),
    ('target_description', 'Target Description', 40),
    ('session_media_count', 'Media Files', 12),
]

# Add image column if embedding images
if EMBED_IMAGES:
    DATA_COLUMNS.append(('target_image', 'Target Image', MAX_IMAGE_WIDTH / 7))

# Write headers
for col_idx, (key, header, width) in enumerate(DATA_COLUMNS, 1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    ws.column_dimensions[get_column_letter(col_idx)].width = width

# Freeze header row
ws.freeze_panes = 'A2'

print(f"✅ Created workbook with {len(DATA_COLUMNS)} columns")


In [ ]:
# Populate spreadsheet with data

print("📥 Populating spreadsheet with data...")
if EMBED_IMAGES:
    print("   (Downloading images - this may take a while)\n")

# Track statistics
stats = {
    'images_embedded': 0,
    'images_failed': 0,
}

# Get column indices
col_indices = {key: idx + 1 for idx, (key, _, _) in enumerate(DATA_COLUMNS)}

# Default row height for image rows (in points)
IMAGE_ROW_HEIGHT = MAX_IMAGE_HEIGHT * 0.75

for row_idx, session in enumerate(flattened_sessions, 2):
    # Progress indicator
    if row_idx % 20 == 0 or row_idx == 2:
        print(f"   Processing row {row_idx-1}/{len(flattened_sessions)}...")
    
    # Set row height for images
    if EMBED_IMAGES:
        ws.row_dimensions[row_idx].height = IMAGE_ROW_HEIGHT
    
    # Write data columns
    for key, _, _ in DATA_COLUMNS:
        if key == 'target_image':
            continue  # Handle image separately
            
        col_idx = col_indices[key]
        value = session.get(key)
        
        # Format boolean values
        if isinstance(value, bool):
            value = "Yes" if value else "No"
        
        ws.cell(row=row_idx, column=col_idx, value=value)
    
    # Embed target image if enabled
    if EMBED_IMAGES and 'target_image' in col_indices:
        target_image_url = session.get('target_image_url')
        if target_image_url:
            image_bytes = download_image(target_image_url)
            if image_bytes:
                xl_img = create_scaled_xl_image(image_bytes, MAX_IMAGE_WIDTH, MAX_IMAGE_HEIGHT)
                if xl_img:
                    cell_ref = f"{get_column_letter(col_indices['target_image'])}{row_idx}"
                    ws.add_image(xl_img, cell_ref)
                    stats['images_embedded'] += 1
                else:
                    stats['images_failed'] += 1
            else:
                stats['images_failed'] += 1

print(f"\n📊 Processing complete!")
print(f"   Sessions processed: {len(flattened_sessions)}")
if EMBED_IMAGES:
    print(f"   Images embedded: {stats['images_embedded']}")
    print(f"   Images failed: {stats['images_failed']}")


In [ ]:
# Save the workbook

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"research_sessions_{timestamp}.xlsx"
output_path = OUTPUT_DIR / output_filename

print(f"💾 Saving workbook to: {output_path}")
wb.save(output_path)

# Verify file was created
if output_path.exists():
    file_size = output_path.stat().st_size
    print(f"\n✅ Successfully created: {output_filename}")
    print(f"📁 Location: {output_path.absolute()}")
    print(f"💾 File size: {file_size / (1024*1024):.2f} MB")
    print(f"📋 Rows: {len(flattened_sessions)} sessions")
    print(f"📊 Columns: {len(DATA_COLUMNS)}")
else:
    print("❌ Failed to create file")


## Done! 🎉

Your Excel file has been created in the `output` folder.

### Column Reference

| Column | Description |
|--------|-------------|
| Session ID | Unique identifier for the session |
| User ID | Unique identifier for the user |
| Display Name | User's display name |
| Tasking Time | When the user started the session (received the target) |
| Submission Time | When the user submitted their session |
| Target Coordinate | The random coordinate assigned to the target |
| Weekly Target ID | If this was a weekly community target, its ID |
| Is Public | Whether the session is publicly visible |
| Is Low Value | Whether the session was flagged as low-value by AI |
| Blockchain Verified | Whether the session was verified on Solana blockchain |
| Self Score | User's self-assessment score (1-7) |
| Community Score Avg | Average score from community ratings |
| CJ Rank | Comparative judging rank (1 = best match) |
| P-Value | Statistical significance |
| Decoy IDs | Comma-separated list of decoy target IDs used in comparative judging |
| Target Description | Description of the remote viewing target |
| Target Image | Embedded image of the target (if enabled) |

### Notes

- Image URLs expire after 1 hour (signed URLs from storage)
- To re-run with fresh URLs, just run the notebook again
- Set `EMBED_IMAGES = False` for faster export without images
